# 02 Vision and CLIP on building photos

Zero-shot, nothing trained: Grounding DINO finds roof, windows and dormers from words, SAM 2.1 cuts the roof out, Depth Anything gives a depth map, CLIP names the roof type and searches the photos by text. Photos: Real2Twin building01 (already on this server). Output: /workspace/usable-space/out/ai/

In [ ]:
import json, time, inspect, numpy as np, torch
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
OUT = Path('/workspace/usable-space/out/ai'); OUT.mkdir(parents=True, exist_ok=True)
SRC = sorted(Path('/workspace/data/building01').glob('*.png'))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
free = torch.cuda.mem_get_info()[0] / 2**30 if dev == 'cuda' else 0
print(len(SRC), 'photos', dev, f'{free:.1f} GiB free')
def load(p, w=1024):
    im = Image.open(p).convert('RGB'); s = w / max(im.size)
    return im.resize((int(im.width * s), int(im.height * s)))
PICK = [SRC[1], SRC[11], SRC[21]]
imgs = [load(p) for p in PICK]
print([p.name for p in PICK], imgs[0].size)

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
gp = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-tiny')
gm = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-tiny').to(dev).eval()
PROMPT = 'a roof. a window. a door. a chimney. a dormer.'
dets = []
for im in imgs:
    inp = gp(images=im, text=PROMPT, return_tensors='pt').to(dev)
    with torch.no_grad(): out = gm(**inp)
    kw = dict(threshold=0.30, text_threshold=0.25, target_sizes=[im.size[::-1]])
    try: r = gp.post_process_grounded_object_detection(out, inp.input_ids, **kw)[0]
    except TypeError:
        kw['box_threshold'] = kw.pop('threshold'); r = gp.post_process_grounded_object_detection(out, inp.input_ids, **kw)[0]
    labels = r.get('text_labels', r.get('labels'))
    dets.append([{'label': str(l), 'score': round(float(s), 2), 'box': [round(float(v)) for v in b]} for l, s, b in zip(labels, r['scores'], r['boxes'])])
    print(len(dets[-1]), 'detections', sorted({d['label'] for d in dets[-1]}))
del gm; torch.cuda.empty_cache()

In [ ]:
from transformers import Sam2Processor, Sam2Model
sp = Sam2Processor.from_pretrained('facebook/sam2.1-hiera-tiny')
sm = Sam2Model.from_pretrained('facebook/sam2.1-hiera-tiny').to(dev).eval()
masks = []
for im, ds in zip(imgs, dets):
    roofs = [d for d in ds if 'roof' in d['label']] or ds[:1]
    if not roofs: masks.append(None); continue
    box = max(roofs, key=lambda d: d['score'])['box']
    inp = sp(images=im, input_boxes=[[box]], return_tensors='pt').to(dev)
    with torch.no_grad(): out = sm(**inp, multimask_output=False)
    m = sp.post_process_masks(out.pred_masks.cpu(), inp['original_sizes'])[0][0, 0].numpy() > 0
    masks.append(m); print('roof mask covers', round(100 * m.mean(), 1), '% of the photo')
del sm; torch.cuda.empty_cache()

In [ ]:
from transformers import pipeline
dp = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf', device=0 if dev == 'cuda' else -1)
depths = [np.array(dp(im)['depth']) for im in imgs]
print([d.shape for d in depths]); del dp; torch.cuda.empty_cache()

In [ ]:
from transformers import CLIPModel, CLIPProcessor
cm = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(dev).eval()
cp = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def text_probs(im, prompts):
    inp = cp(text=prompts, images=im, return_tensors='pt', padding=True).to(dev)
    with torch.no_grad(): lg = cm(**inp).logits_per_image[0]
    return torch.softmax(lg, dim=0).cpu().numpy()
ROOF = ['a gable roof', 'a hip roof', 'a flat roof', 'a mansard roof', 'a shed roof', 'a half-hipped roof']
STYLE = ['a half-timbered house', 'a brick house', 'a modern house', 'a stone building', 'a plastered house']
clip = {'roof': [], 'style': []}
for im in imgs:
    clip['roof'].append({p: round(float(v), 3) for p, v in zip(ROOF, text_probs(im, ROOF))})
    clip['style'].append({p: round(float(v), 3) for p, v in zip(STYLE, text_probs(im, STYLE))})
# text search over all photos
embs = []
for p in SRC:
    inp = cp(images=load(p, 512), return_tensors='pt').to(dev)
    with torch.no_grad(): e = cm.get_image_features(**inp)
    e = e if torch.is_tensor(e) else e.pooler_output
    embs.append(torch.nn.functional.normalize(e, dim=-1)[0].cpu())
E = torch.stack(embs)
QUERIES = ['a roof with a dormer window', 'a close-up of a window', 'a brick wall']
search = {}
for q in QUERIES:
    inp = cp(text=[q], return_tensors='pt', padding=True).to(dev)
    with torch.no_grad(): t = cm.get_text_features(**inp)
    t = t if torch.is_tensor(t) else t.pooler_output
    t = torch.nn.functional.normalize(t, dim=-1)[0].cpu()
    sc = (E @ t).numpy(); top = np.argsort(-sc)[:3]
    search[q] = [{'photo': SRC[i].name, 'score': round(float(sc[i]), 3)} for i in top]
print(json.dumps(clip['roof'][0], indent=1)); print(json.dumps(search, indent=1))
del cm; torch.cuda.empty_cache()

In [ ]:
W = 420
def fit(im): return im.resize((W, int(im.height * W / im.width)))
def overlay(im, ds, m):
    base = np.array(im).astype(float)
    if m is not None: base[m] = base[m] * 0.45 + np.array([232, 69, 44]) * 0.55
    o = Image.fromarray(base.astype('uint8')); d = ImageDraw.Draw(o)
    for x in ds:
        c = (20, 22, 28) if 'roof' not in x['label'] else (232, 69, 44)
        d.rectangle(x['box'], outline=c, width=3); d.text((x['box'][0] + 4, x['box'][1] + 2), f"{x['label']} {x['score']}", fill=(255, 255, 255))
    return o
def dmap(dep):
    v = (dep - dep.min()) / (dep.max() - dep.min() + 1e-6); v = (255 * (1 - v)).astype('uint8')
    return Image.fromarray(v).convert('RGB')
rows = [[fit(im), fit(overlay(im, ds, m)), fit(dmap(dep).resize(im.size))] for im, ds, m, dep in zip(imgs, dets, masks, depths)]
H = sum(r[0].height for r in rows)
sheet = Image.new('RGB', (W * 3, H), (255, 255, 255)); y = 0
for r in rows:
    for c, t in enumerate(r): sheet.paste(t, (c * W, y))
    y += r[0].height
sheet.save(OUT / 'ai_montage.jpg', quality=85)
for i, p in enumerate(SRC): pass
thumbs = Image.new('RGB', (W * 3, 240 * 3))
for q_i, q in enumerate(QUERIES):
    for k, hit in enumerate(search[q]):
        t = load(Path('/workspace/data/building01') / hit['photo'], 420); t = t.crop((0, 0, 420, min(t.height, 240))); thumbs.paste(t, (k * W, q_i * 240))
thumbs.save(OUT / 'clip_search.jpg', quality=85)
(OUT / 'ai_results.json').write_text(json.dumps({'photos': [p.name for p in PICK], 'detections': dets, 'clip': clip, 'search': search, 'queries': QUERIES}, indent=1), encoding='utf-8')
print('written', sorted(p.name for p in OUT.iterdir()), (OUT / 'ai_montage.jpg').stat().st_size // 1024, 'KB')